# Rough Heston QIPC Demo

This notebook demonstrates European call pricing under the rough Heston model using the Quadratic-Implicit Fractional Adams Predictor-Corrector method. The first cell installs the package so it can run directly in Google Colab.

In [ ]:
# Cell 1 — install
!pip install rough-heston-qipc --quiet

In [ ]:
# Cell 2 — imports
import time
import numpy as np
import matplotlib.pyplot as plt

from rough_heston_qipc import (
    RoughHestonModel,
    RoughHestonParams,
    timed_price,
)

## 1. Simple worked example

We compute one European call price using the default model parameters and a moderate numerical grid.

In [ ]:
model = RoughHestonModel(RoughHestonParams())
price = model.calculate(NOuter=50, NInner=500)
print(f"implicit price: {price:.12f}")

## 2. Runtime measurement

In [ ]:
price, elapsed = timed_price(NOuter=50, NInner=500)
print(f"price: {price:.12f}")
print(f"elapsed: {elapsed:.6f} seconds")

## 3. Convergence as the number of time steps increases

The fractional Adams recursion is controlled by `NInner`. Larger values are slower but should produce a more stable price sequence.

In [ ]:
n_inner_grid = np.array([50, 100, 200, 400, 800])
prices = []
times = []

for n_inner in n_inner_grid:
    start = time.perf_counter()
    p = model.calculate(NOuter=50, NInner=int(n_inner))
    elapsed = time.perf_counter() - start
    prices.append(p)
    times.append(elapsed)
    print(f"NInner={n_inner:4d}, price={p:.12f}, time={elapsed:.4f}s")

prices = np.array(prices)
times = np.array(times)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(n_inner_grid, prices, marker="o")
plt.xlabel("NInner")
plt.ylabel("Option price")
plt.title("Convergence of quadratic-implicit solver")
plt.grid(True)
plt.show()

## 4. Compare with explicit predictor-corrector baseline

The package includes the original explicit corrector as a benchmark. The comparison below reports price, runtime, and absolute price difference.

In [ ]:
comparison_grid = [(25, 100), (35, 200), (50, 500)]

for n_outer, n_inner in comparison_grid:
    start = time.perf_counter()
    p_new = model.calculate(n_outer, n_inner, method="implicit")
    t_new = time.perf_counter() - start

    start = time.perf_counter()
    p_old = model.calculate(n_outer, n_inner, method="explicit")
    t_old = time.perf_counter() - start

    print(
        f"NOuter={n_outer:3d}, NInner={n_inner:4d} | "
        f"new={p_new:.12f} ({t_new:.4f}s), "
        f"explicit={p_old:.12f} ({t_old:.4f}s), "
        f"abs diff={abs(p_new - p_old):.3e}"
    )

## 5. Parameter stress example

Here we vary the roughness parameter `alpha` while holding the remaining parameters fixed.

In [ ]:
alpha_grid = np.array([0.55, 0.60, 0.65, 0.70, 0.75])
alpha_prices = []

for alpha in alpha_grid:
    params = RoughHestonParams(alpha=float(alpha))
    alpha_model = RoughHestonModel(params)
    alpha_prices.append(alpha_model.calculate(50, 400))

plt.figure(figsize=(7, 4))
plt.plot(alpha_grid, alpha_prices, marker="o")
plt.xlabel("alpha")
plt.ylabel("Option price")
plt.title("Sensitivity to roughness parameter")
plt.grid(True)
plt.show()